# Chapter 12: Floer Homology

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Chapter 12, printed pp. 487-530; PDF pp. 502-545. Sections: 12.1-12.7.

    ## Chapter Goal

    Floer cochain complex, ring structure, Poincare duality, spectral invariants, Seidel representation, Donaldson's quantum category, and vortex equations.

    The guiding question is:

    > How does one build homology from Hamiltonian trajectories in the same spirit as curve-counting moduli spaces?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| periodic orbit | critical point of action | action value |
| Floer trajectory | solution on a cylinder | energy equals action drop |
| differential | count of index-one trajectories | boundary squared zero |
| product | pair-of-pants moduli count | degree and associativity |
| spectral invariant | action level selected by homology class | filtration bound |


## Standalone Reading Guide

    The final chapter surveys Floer homology as a cousin of the Gromov-Witten story. Instead of closed curves in a target, the generators are Hamiltonian periodic orbits. The differential counts solutions of a perturbed Cauchy-Riemann equation on the cylinder connecting one orbit to another. Energy is controlled by action drop, so compactness and breaking replace the stable-map boundary pictures from earlier chapters.

The algebraic signature is d squared equals zero. The boundary of a one-dimensional moduli space of connecting trajectories consists of broken pairs, and those pairs are counted twice over Z/2 or cancel with signs in the oriented theory. Products arise from pair-of-pants domains and connect Floer homology back to quantum cohomology through comparison maps.

The notebook uses a Morse-theory shadow on the circle because the full analytical setup would be too large for a small executable model. Two gradient trajectories from a maximum to a minimum cancel over Z/2, so the differential vanishes and its square vanishes. The picture emphasizes action filtration and trajectory counting, the ingredients that persist in Hamiltonian Floer theory.

    ## Topics In This Notebook

    - Hamiltonian action functional and periodic orbits
- Floer differential from connecting trajectories
- pair-of-pants product and relation to quantum cohomology
- spectral invariants and action filtrations
- Seidel representation, quantum categories, and vortex directions

    ## Visualization Storyboard

    - A toy Morse-Floer complex on a circle shows generators, action levels, and cancelling trajectories.
- A dependency graph connects action, trajectories, compactness, and d^2=0.
- A ledger checks the boundary-squared-zero identity over Z/2.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'chapter-12'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['periodic orbit', 'Floer trajectory', 'differential', 'product', 'spectral invariant']
CONCEPT_EDGES = [('periodic orbit', 'Floer trajectory'), ('Floer trajectory', 'differential'), ('differential', 'product'), ('product', 'spectral invariant')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=24, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: Floer Homology')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '487-530',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in Floer Homology. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
theta = np.linspace(0, 2 * np.pi, 300)
height = np.cos(theta)
crit = {"max": (0.0, 1.0, 1), "min": (np.pi, -1.0, 0)}
boundary_mod2 = np.array([[0, 0], [0, 0]], dtype=int)
trajectory_count_max_to_min = 2
boundary_mod2[1, 0] = trajectory_count_max_to_min % 2
d_squared = (boundary_mod2 @ boundary_mod2) % 2

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(theta, height, color="#2c7fb8")
ax.scatter([crit["max"][0], crit["min"][0]], [crit["max"][1], crit["min"][1]], s=90, color=["#d95f0e", "#31a354"], zorder=3)
ax.annotate("two trajectories cancel mod 2", xy=(np.pi / 2, 0), xytext=(1.2, 0.55), arrowprops={"arrowstyle": "->"})
ax.annotate("", xy=(np.pi, -1), xytext=(0, 1), arrowprops={"arrowstyle": "->", "connectionstyle": "arc3,rad=0.35", "color": "#555"})
ax.annotate("", xy=(np.pi, -1), xytext=(2 * np.pi, 1), arrowprops={"arrowstyle": "->", "connectionstyle": "arc3,rad=-0.35", "color": "#555"})
ax.set_xlabel("circle coordinate")
ax.set_ylabel("action/Morse value")
ax.set_title("Toy Floer complex: action drop and broken-boundary cancellation")
ax.grid(True, alpha=0.25)
fig_path = save_matplotlib(fig, UNIT, "figures", "floer-toy-action-complex.png")
plt.close(fig)

check = {
    "trajectory_count_max_to_min": trajectory_count_max_to_min,
    "boundary_matrix_mod2": boundary_mod2.tolist(),
    "d_squared_mod2": d_squared.tolist(),
    "passed": bool(np.all(d_squared == 0)),
}
check_path = save_json(check, UNIT, "checks", "floer-boundary-squared-checks.json")
display_artifact(fig_path, width=720)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'periodic orbit', 'computational_object': 'critical point of action', 'check': 'action value'}, {'item': 'Floer trajectory', 'computational_object': 'solution on a cylinder', 'check': 'energy equals action drop'}, {'item': 'differential', 'computational_object': 'count of index-one trajectories', 'check': 'boundary squared zero'}, {'item': 'product', 'computational_object': 'pair-of-pants moduli count', 'check': 'degree and associativity'}, {'item': 'spectral invariant', 'computational_object': 'action level selected by homology class', 'check': 'filtration bound'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Change the coefficient system from mod 2 to signed integers by assigning opposite signs to the two trajectories. The cancellation mechanism should remain visible.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - Floer generators are periodic orbits, and differentials count connecting cylinders.
- Compactness appears through broken trajectories rather than only through bubbles.
- The relation to quantum cohomology is mediated by products and comparison maps.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'floer-toy-action-complex.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'floer-boundary-squared-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'floer-boundary-squared-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
